# 메뉴 크롤링

## 1. 메가커피

In [1]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
from rapidfuzz import fuzz

print("✅ 모든 라이브러리 정상 설치")

C:\Users\pyk71\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\pyk71\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


✅ 모든 라이브러리 정상 설치


In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://www.mega-mgccoffee.com/menu/?menu_category1=1&menu_category2=1"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("상태 코드:", response.status_code)
print("페이지 길이:", len(response.text))

상태 코드: 200
페이지 길이: 34219


In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")

driver = webdriver.Chrome(service=service, options=options)
driver.get("https://www.google.com")
print(driver.title)
driver.quit()

Google


In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=service, options=options)

BASE_URL = "https://www.mega-mgccoffee.com/menu/?menu_category1=1&menu_category2=1"

CATEGORY_MAP = {
    '1': '커피',
    '2': '티',
    '3': '에이드&주스',
    '4': '스무디&프라페',
    '5': '디카페인',
    '6': '음료',
}
COFFEE_CATEGORIES = {'1', '5'}

all_items = []

def parse_page(soup, category, category_name):
    items = []
    menu_list = soup.find('ul', id='menu_list')
    if not menu_list:
        return items
    for li in menu_list.find_all('li'):
        try:
            name_tag = li.find('b')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            eng_div = li.find('div', class_='cont_text_info')
            eng_name = eng_div.find('div', class_='text').get_text(strip=True) if eng_div else ''

            label = li.find('div', class_='cont_gallery_list_label')
            temp = label.get_text(strip=True) if label else ''

            items.append({
                'brand': '메가커피',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'category': category_name,
                'is_coffee': category in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

def get_total_pages(soup):
    pages = soup.select('#board_page li a.board_page_link')
    page_numbers = [int(p.get_text(strip=True)) for p in pages if p.get_text(strip=True).isdigit()]
    return max(page_numbers) if page_numbers else 1

# 카테고리별 수집
for cat_value, cat_name in CATEGORY_MAP.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get(BASE_URL)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "menu_list")))
    time.sleep(2)

    # 체크박스 클릭
    checkbox = driver.find_element(By.CSS_SELECTOR, f'input[name="list_checkbox"][value="{cat_value}"]')
    driver.execute_script("arguments[0].click();", checkbox)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    total_pages = get_total_pages(soup)
    print(f"  총 {total_pages}페이지")

    page_items = parse_page(soup, cat_value, cat_name)
    all_items.extend(page_items)
    print(f"  1페이지: {len(page_items)}개")

    for page in range(2, total_pages + 1):
        try:
            btn = driver.find_element(By.CSS_SELECTOR, f'a.board_page_link[data-page="{page}"]')
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            page_items = parse_page(soup, cat_value, cat_name)
            all_items.extend(page_items)
            print(f"  {page}페이지: {len(page_items)}개")
        except Exception as e:
            print(f"  {page}페이지 에러: {e}")
            continue

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('mega_coffee_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 저장 → mega_coffee_menu.csv")
print(df.head(10))
print(f"\n카테고리별 수집 수:\n{df['category'].value_counts()}")


▶ 커피 수집 시작
  총 3페이지
  1페이지: 20개
  2페이지: 20개
  3페이지: 1개

▶ 티 수집 시작
  총 2페이지
  1페이지: 20개
  2페이지: 5개

▶ 에이드&주스 수집 시작
  총 1페이지
  1페이지: 12개

▶ 스무디&프라페 수집 시작
  총 1페이지
  1페이지: 20개

▶ 디카페인 수집 시작
  총 2페이지
  1페이지: 20개
  2페이지: 11개

▶ 음료 수집 시작
  총 1페이지
  1페이지: 19개

✅ 완료! 총 148개 저장 → mega_coffee_menu.csv
  brand           name                                      name_en temp  \
0  메가커피     초코젤라또 말차라떼                Choco-gelato Matcha Tea Latte  ICE   
1  메가커피     (HOT)헛개리카노                Oriental Raisin-Tea Americano  HOT   
2  메가커피     (ICE)헛개리카노                Oriental Raisin-Tea Americano  ICE   
3  메가커피        왕메가카페라떼                         BIG MEGA Caffe Latte  ICE   
4  메가커피       왕메가헛개리카노      BIG MEGA  Oriental Raisin-Tea Americano  ICE   
5  메가커피        할메가미숫커피  MEGA MGC Mix Coffee Blend with Grain Powder  ICE   
6  메가커피  라이트 바닐라 아몬드라떼                   Light vanilla almond latte  ICE   
7  메가커피           연유라떼                         Condensed Milk Latte  ICE   
8  메가커피          할메가커피

## 2. 컴포즈

In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://composecoffee.com/menu/category/185"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("상태 코드:", response.status_code)
print("페이지 길이:", len(response.text))

상태 코드: 200
페이지 길이: 56664


In [2]:
soup = BeautifulSoup(response.text, 'html.parser')

tags = soup.find_all(['div', 'li', 'ul'], class_=True)

seen = set()
for tag in tags:
    cls = ' '.join(tag.get('class', []))
    if cls not in seen:
        seen.add(cls)
        text = tag.get_text(strip=True)[:30]
        print(f"<{tag.name} class='{cls}'> | {text}")

<div class='hd_bt'> | BRANDABOUT US비터홀릭FRANCHISE가맹문의
<div class='hd_con'> | BRANDABOUT US비터홀릭FRANCHISE가맹문의
<div class='mn'> | BRANDABOUT US비터홀릭FRANCHISE가맹문의
<div class='hd-wrap'> | 
<ul class='lang-ul'> | 
<ul class='mem-ul'> | 
<ul class='sns-ul'> | 
<div class='hd-btn'> | 
<div class='mt-menu'> | BRANDABOUT US비터홀릭FRANCHISE가맹문의
<ul class='mo_mn_mn'> | BRANDABOUT US비터홀릭FRANCHISE가맹문의
<div class='wrapper'> | 본문으로 이동하기MENUBRANDFRANCHISEMEN
<div class='subtop'> | MENUBRANDFRANCHISEMENUSTORENEW
<div class='st-caption'> | 
<div class='navmn-sub second'> | MENUBRANDFRANCHISEMENUSTORENEW
<div class='navmn-con'> | MENUBRANDFRANCHISEMENUSTORENEW
<ul class='navmn'> | MENUBRANDFRANCHISEMENUSTORENEW
<li class='home'> | 
<li class='depth1 dropdown'> | MENUBRANDFRANCHISEMENUSTORENEW
<div class='dropdown-content'> | BRANDFRANCHISEMENUSTORENEWSPar
<div class='art_con'> | 전체컴포즈 콤보시즌한정커피 · 더치논커피 라떼프라페 ·
<div class='restboard'> | 전체컴포즈 콤보시즌한정커피 · 더치논커피 라떼프라페 ·
<ul class='nav nav-pills nav-sm nav-fill mb-5

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

# 메인 메뉴 페이지에서 카테고리 URL 자동 추출
res = requests.get("https://composecoffee.com/menu/category/185", headers=headers)
soup = BeautifulSoup(res.text, 'html.parser')

# 카테고리 탭 목록 찾기
category_tabs = soup.select('ul.nav-pills li.nav-item a')

categories = {}
for tab in category_tabs:
    name = tab.get_text(strip=True)
    href = tab.get('href', '')
    if '/menu/category/' in href:
        full_url = 'https://composecoffee.com' + href if href.startswith('/') else href
        categories[name] = full_url

print("수집된 카테고리 목록:")
for name, url in categories.items():
    print(f"  {name} → {url}")

C:\Users\pyk71\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\pyk71\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


수집된 카테고리 목록:
  컴포즈 콤보 → https://composecoffee.com/menu/category/207002
  시즌한정 → https://composecoffee.com/menu/category/192677
  커피 · 더치 → https://composecoffee.com/menu/category/185
  논커피 라떼 → https://composecoffee.com/menu/category/187
  프라페 · 스무디 → https://composecoffee.com/menu/category/192
  밀크쉐이크 → https://composecoffee.com/menu/category/193
  에이드 · 주스 → https://composecoffee.com/menu/category/188
  티 → https://composecoffee.com/menu/category/191
  디저트 → https://composecoffee.com/menu/category/339
  MD상품 → https://composecoffee.com/menu/category/98609


In [4]:
# 커피로 볼 카테고리 지정 (나머지는 False)
COFFEE_CATEGORIES = {'커피 · 더치'}

# 수집 제외할 카테고리 (음료 아닌 것들)
EXCLUDE_CATEGORIES = {'컴포즈 콤보', '디저트', 'MD상품'}

# 최종 수집 대상 필터링
target_categories = {
    name: url 
    for name, url in categories.items() 
    if name not in EXCLUDE_CATEGORIES
}

print("수집 대상 카테고리:")
for name, url in target_categories.items():
    is_coffee = name in COFFEE_CATEGORIES
    print(f"  {name} → is_coffee: {is_coffee}")

수집 대상 카테고리:
  시즌한정 → is_coffee: False
  커피 · 더치 → is_coffee: True
  논커피 라떼 → is_coffee: False
  프라페 · 스무디 → is_coffee: False
  밀크쉐이크 → is_coffee: False
  에이드 · 주스 → is_coffee: False
  티 → is_coffee: False


In [26]:
def parse_page(soup, cat_name):
    items = []
    boxes = soup.find_all('div', class_='itemBox')
    
    for box in boxes:
        try:
            name = box.find('div', class_='header').get_text(strip=True)
            items.append({
                'brand': '컴포즈커피',
                'name': name,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [27]:
all_items = []

for cat_name, base_url in target_categories.items():
    print(f"\n▶ {cat_name} 수집 시작")
    page = 1

    while True:
        url = f"{base_url}?page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        items = parse_page(soup, cat_name)
        if not items:
            print(f"  {page}페이지 데이터 없음 → 종료")
            break

        all_items.extend(items)
        print(f"  {page}페이지: {len(items)}개")

        # 다음 페이지 확인
        next_btn = soup.select_one('li.page-item:not(.disabled) a[aria-label="Next"]')
        if not next_btn:
            break
        page += 1
        time.sleep(1)

print(f"\n총 {len(all_items)}개 수집 완료")


▶ 시즌한정 수집 시작
  1페이지: 20개
  2페이지: 13개
  3페이지 데이터 없음 → 종료

▶ 커피 · 더치 수집 시작
  1페이지: 20개
  2페이지: 11개
  3페이지 데이터 없음 → 종료

▶ 논커피 라떼 수집 시작
  1페이지: 18개
  2페이지 데이터 없음 → 종료

▶ 프라페 · 스무디 수집 시작
  1페이지: 12개
  2페이지 데이터 없음 → 종료

▶ 밀크쉐이크 수집 시작
  1페이지: 7개
  2페이지 데이터 없음 → 종료

▶ 에이드 · 주스 수집 시작
  1페이지: 12개
  2페이지 데이터 없음 → 종료

▶ 티 수집 시작
  1페이지: 20개
  2페이지: 11개
  3페이지 데이터 없음 → 종료

총 144개 수집 완료


In [29]:
df = pd.DataFrame(all_items)
df.to_csv('compose_coffee_menu.csv', index=False, encoding='utf-8-sig')

print(f"✅ 저장 완료 → compose_coffee_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → compose_coffee_menu.csv

카테고리별 수집 수:
category
시즌한정         33
커피 · 더치      31
티            31
논커피 라떼       18
프라페 · 스무디    12
에이드 · 주스     12
밀크쉐이크         7
Name: count, dtype: int64

샘플 데이터:
   brand                 name category  is_coffee price
0  컴포즈커피  ICE THE CITY 생초콜릿라떼     시즌한정      False  None
1  컴포즈커피   ICE THE CITY 올데이오트     시즌한정      False  None
2  컴포즈커피   HOT THE CITY 올데이오트     시즌한정      False  None
3  컴포즈커피        ICE 에어리 아메리카노     시즌한정      False  None
4  컴포즈커피              ICE 매샷추     시즌한정      False  None
5  컴포즈커피           말차샷 유자 스무디     시즌한정      False  None
6  컴포즈커피         ICE 크림 말차 라떼     시즌한정      False  None
7  컴포즈커피        ICE 어센틱 말차 라떼     시즌한정      False  None
8  컴포즈커피        HOT 어센틱 말차 라떼     시즌한정      False  None
9  컴포즈커피       ICE 에어레이팅 꿀 말차     시즌한정      False  None


## 3. 빽다방

In [9]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

url = "https://paikdabang.com/menu/menu_coffee/"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 303009


In [10]:
soup = BeautifulSoup(res.text, 'html.parser')

# 카테고리 탭 링크 찾기
category_tabs = soup.select('a[href*="/menu/"]')

seen = set()
for tab in category_tabs:
    href = tab.get('href', '')
    name = tab.get_text(strip=True)
    if href not in seen and name:
        seen.add(href)
        print(f"  {name} → {href}")

  메뉴 → https://paikdabang.com/menu/menu_new/
  커피 → https://paikdabang.com/menu/menu_coffee/
  음료 → https://paikdabang.com/menu/menu_drink/
  아이스크림/디저트 → https://paikdabang.com/menu/menu_dessert/
  빽스치노 → https://paikdabang.com/menu/menu_ccino/


In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

# 커피로 볼 카테고리
COFFEE_CATEGORIES = {'커피'}

# 수집 제외 카테고리
EXCLUDE_CATEGORIES = {'아이스크림/디저트'}

CATEGORIES = {
    '커피': 'https://paikdabang.com/menu/menu_coffee/',
    '음료': 'https://paikdabang.com/menu/menu_drink/',
    '빽스치노': 'https://paikdabang.com/menu/menu_ccino/',
}

print("✅ 설정 완료")

✅ 설정 완료


In [21]:
import re

def parse_paikdabang(soup, cat_name):
    items = []
    menu_list = soup.find('div', class_='menu_list')
    if not menu_list:
        return items

    for li in menu_list.find_all('li'):
        try:
            # 메뉴명
            name = li.find('p', class_='menu_tit')
            if not name:
                continue
            name = name.get_text(strip=True)

            # 영문명
            eng = li.find('div', class_='menu_tit2')
            eng_name = eng.get_text(strip=True) if eng else ''

            # ICE / HOT (메뉴명에서 추출)
            if '(ICED)' in name or 'ICED' in name.upper():
                temp = 'ICE'
            elif '(HOT)' in name or 'HOT' in name.upper():
                temp = 'HOT'
            else:
                temp = ''

            # 사이즈 (oz 추출)
            size_tag = li.find('p', class_='menu_ingredient_basis')
            size = ''
            if size_tag:
                # 여러 개일 수 있으니 전체 탐색
                for p in li.find_all('p', class_='menu_ingredient_basis'):
                    text = p.get_text(strip=True)
                    match = re.search(r'(\d+)\s*oz', text)
                    if match:
                        size = match.group(1) + 'oz'
                        break

            # 커피 여부 (고카페인 or 카테고리 기준)
            txt = li.find('p', class_='txt')
            txt_text = txt.get_text(strip=True) if txt else ''
            is_coffee = cat_name in COFFEE_CATEGORIES or '고카페인' in txt_text

            items.append({
                'brand': '빽다방',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [22]:
all_items = []

for cat_name, url in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    items = parse_paikdabang(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")
    time.sleep(1)

print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  108개 수집

▶ 음료 수집 시작
  143개 수집

▶ 빽스치노 수집 시작
  18개 수집

총 269개 수집 완료


In [23]:
df = pd.DataFrame(all_items)
df.to_csv('paikdabang_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → paikdabang_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → paikdabang_menu.csv

카테고리별 수집 수:
category
음료      143
커피      108
빽스치노     18
Name: count, dtype: int64

샘플 데이터:
  brand              name                        name_en temp  size category  \
0   빽다방   에어폼 아메리카노(ICED)             AIR FOAM AMERICANO  ICE  24oz       커피   
1   빽다방           더블에스프레소                double espresso                  커피   
2   빽다방      디카페인 더블에스프레소          DECAF DOUBLE ESPRESSO                  커피   
3   빽다방        아메리카노(HOT)                      AMERICANO  HOT  16oz       커피   
4   빽다방       아메리카노(ICED)                      AMERICANO  ICE  24oz       커피   
5   빽다방   디카페인 아메리카노(HOT)                DECAF AMERICANO  HOT  16oz       커피   
6   빽다방  디카페인 아메리카노(ICED)                DECAF AMERICANO  ICE  24oz       커피   
7   빽다방     레드불 꿀샷추(ICED)  RED BULL with HONEY, ESPRESSO  ICE  24oz       커피   
8   빽다방         원조커피(HOT)          original mixed coffee  HOT  16oz       커피   
9   빽다방        원조커피(ICED)          ORIGINAL MIXED COFFEE  ICE  24oz       커피 

## 4. 이디야

In [30]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

url = "https://www.ediya.com/contents/drink.html"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 76238


In [34]:
menu_ul = soup.find('ul', id='menu_ul')
items = menu_ul.find_all('li') if menu_ul else []
print(f"현재 로딩된 메뉴 수: {len(items)}개")

# 더보기 버튼 있는지 확인
more_btn = soup.find('div', class_='con_btn')
print(f"더보기 버튼: {'있음' if more_btn else '없음'}")

# hidden input 페이지 값 확인
page_input = soup.find('input', id='menu_page')
print(f"현재 페이지 값: {page_input.get('value') if page_input else '없음'}")

현재 로딩된 메뉴 수: 8개
더보기 버튼: 있음
현재 페이지 값: 2


In [35]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# 커피로 볼 카테고리
COFFEE_CATEGORIES = {'COFFEE', 'DECAF'}

# 수집 제외 카테고리
EXCLUDE_CATEGORIES = {'ICE CREAM', 'TOPPING', 'RTD'}

CATEGORIES = {
    'COFFEE': '12',
    'BEVERAGE': '13',
    'BLENDING TEA': '14',
    'FLATCCINO': '15',
    'SHAKE & ADE': '16',
    'ICE FLAKES': '71',
    'DECAF': '155',
}

print("✅ 설정 완료")

✅ 설정 완료


In [40]:
def parse_ediya(soup, cat_name):
    items = []
    menu_ul = soup.find('ul', id='menu_ul')
    if not menu_ul:
        return items

    for li in menu_ul.find_all('li'):
        try:
            # 메뉴명 (menu_tt에서)
            name_tag = li.find('div', class_='menu_tt')
            if not name_tag:
                continue
            span = name_tag.find('span')
            name = span.get_text(strip=True) if span else name_tag.get_text(strip=True)

            # 영문명 (h2 안의 span)
            h2 = li.find('h2')
            eng_name = ''
            if h2:
                span = h2.find('span')
                eng_name = span.get_text(strip=True) if span else ''

            # temp (메뉴명에서 추출)
            if 'ICED' in name.upper():
                temp = 'ICE'
            elif 'HOT' in name.upper():
                temp = 'HOT'
            else:
                temp = ''

            # 사이즈 (메뉴명 앞 (L), (EX) + pro_size ml)
            size_label = ''
            if name.startswith('(L)'):
                size_label = 'L'
            elif name.startswith('(EX)'):
                size_label = 'EX'

            size_tag = li.find('div', class_='pro_size')
            size_ml = ''
            if size_tag:
                match = re.search(r'(\d+)ml', size_tag.get_text())
                size_ml = match.group(1) + 'ml' if match else ''

            size = f"{size_label} {size_ml}".strip()

            # 커피 여부
            is_coffee = cat_name in COFFEE_CATEGORIES

            items.append({
                'brand': '이디야',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [41]:
all_items = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, cat_value in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://www.ediya.com/contents/drink.html")
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "menu_ul"))
    )
    time.sleep(2)

    # 카테고리 체크박스 클릭
    checkbox = driver.find_element(By.CSS_SELECTOR, f'input[value="{cat_value}"]')
    driver.execute_script("arguments[0].click();", checkbox)
    time.sleep(2)

    # 더보기 버튼 계속 클릭
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'div.con_btn a')
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = parse_ediya(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ COFFEE 수집 시작
  66개 수집

▶ BEVERAGE 수집 시작
  66개 수집

▶ BLENDING TEA 수집 시작
  74개 수집

▶ FLATCCINO 수집 시작
  20개 수집

▶ SHAKE & ADE 수집 시작
  24개 수집

▶ ICE FLAKES 수집 시작
  0개 수집

▶ DECAF 수집 시작
  65개 수집

총 315개 수집 완료


In [43]:
df = pd.DataFrame(all_items)
df.to_csv('ediya_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → ediya_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → ediya_menu.csv

카테고리별 수집 수:
category
BLENDING TEA    74
COFFEE          66
BEVERAGE        66
DECAF           65
SHAKE & ADE     24
FLATCCINO       20
Name: count, dtype: int64

샘플 데이터:
  brand                  name                           name_en temp  \
0   이디야      (L) HOT 카페 아메리카노            (L) HOT Cafe Americano  HOT   
1   이디야     (EX) HOT 카페 아메리카노           (EX) HOT Cafe Americano  HOT   
2   이디야     (L) ICED 카페 아메리카노           (L) ICED Cafe Americano  ICE   
3   이디야    (EX) ICED 카페 아메리카노          (EX) ICED Cafe Americano  ICE   
4   이디야          (L) HOT 달달커피              (L) HOT Mixed coffee  HOT   
5   이디야         (EX) HOT 달달커피             (EX) HOT Mixed coffee  HOT   
6   이디야     (L) HOT 제로슈가 달달커피   (L) HOT Zero sugar Mixed Coffee  HOT   
7   이디야    (EX) HOT 제로슈가 달달커피  (EX) HOT Zero sugar Mixed Coffee  HOT   
8   이디야   (L) ICED 헤이즐넛 아메리카노       (L) ICED Hazelnut Americano  ICE   
9   이디야  (EX) ICED 헤이즐넛 아메리카노      (EX) ICED Hazelnut Americano  ICE   

       size

## 5. 투썸

In [44]:
url = "https://mo.twosome.co.kr/mn/menuInfoList.do"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 31149


In [50]:
COFFEE_SUBCATEGORIES = {'커피'}
EXCLUDE_SUBCATEGORIES = {'아이스크림/빙수', 'NEW'}

SUBCATEGORIES = {
    '커피': '01',
    '음료': '02',
    '티/티라떼': '03',
}

print("✅ 설정 완료")

✅ 설정 완료


In [51]:
def parse_twosome(soup, cat_name):
    items = []
    menu_list = soup.find('ul', class_='ui-goods-list-default')
    if not menu_list:
        return items

    for li in menu_list.find_all('li'):
        try:
            name_tag = li.find('p', class_='menu-title')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            items.append({
                'brand': '투썸플레이스',
                'name': name,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_SUBCATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [52]:
all_items = []
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://mo.twosome.co.kr/mn/menuInfoList.do")
time.sleep(3)

# 커피/음료 대카테고리 클릭
main_tab = driver.find_element(By.CSS_SELECTOR, 'a.tab[grtval="1"]')
driver.execute_script("arguments[0].click();", main_tab)
time.sleep(2)

for cat_name, cat_value in SUBCATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")

    # 서브카테고리 클릭
    sub_tab = driver.find_element(By.CSS_SELECTOR, f'a.tab[midval="{cat_value}"]')
    driver.execute_script("arguments[0].click();", sub_tab)
    time.sleep(2)

    # 더보기 버튼 있으면 계속 클릭
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'a.btn-more')
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = parse_twosome(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  28개 수집

▶ 음료 수집 시작
  29개 수집

▶ 티/티라떼 수집 시작
  19개 수집

총 76개 수집 완료


In [53]:
df = pd.DataFrame(all_items)
df.to_csv('twosome_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → twosome_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → twosome_menu.csv

카테고리별 수집 수:
category
음료       29
커피       28
티/티라떼    19
Name: count, dtype: int64

샘플 데이터:
    brand          name category  is_coffee price
0  투썸플레이스  생크림 말차 카페 라떼       커피       True  None
1  투썸플레이스     생크림 카페 라떼       커피       True  None
2  투썸플레이스     생크림 아메리카노       커피       True  None
3  투썸플레이스  피스타치오 초콜릿 모카       커피       True  None
4  투썸플레이스  디카페인 콜드브루 라떼       커피       True  None
5  투썸플레이스       바닐라빈 라떼       커피       True  None
6  투썸플레이스         아메리카노       커피       True  None
7  투썸플레이스         카페 라떼       커피       True  None
8  투썸플레이스          카푸치노       커피       True  None
9  투썸플레이스        바닐라 라떼       커피       True  None


## 6. 매머드 커피

In [54]:
url = "https://mmthcoffee.com/sub/menu/list_coffee_sub.php?menuType=C"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 21493


In [66]:
COFFEE_CATEGORIES = {'커피', '콜드브루', '32oz'}

CATEGORIES = {
    '32oz': 'O',
    '커피': 'C',
    '콜드브루': 'D',
    '논커피': 'N',
    '티·에이드': 'T',
    '프라페·블렌디드': 'B',
}

BASE_URL = "https://mmthcoffee.com/sub/menu/list_coffee_sub.php?menuType={}"
print("✅ 설정 완료")

✅ 설정 완료


In [72]:
import re

def parse_size_from_popup(driver):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.find('div', class_='i_table')
    temp, size = '', ''
    
    if table:
        # 헤더에서 HOT/ICE 직접 찾기
        headers = table.find_all('th')
        hot_size, ice_size = '', ''
        
        for th in headers:
            text = th.get_text(strip=True)
            hot_match = re.search(r'HOT\((\d+oz)\)', text)
            ice_match = re.search(r'ICE\((\d+oz)\)', text)
            if hot_match:
                hot_size = hot_match.group(1)
            if ice_match:
                ice_size = ice_match.group(1)

        if hot_size and ice_size:
            temp = 'HOT/ICE'
            size = f"HOT({hot_size})/ICE({ice_size})"
        elif hot_size:
            temp = 'HOT'
            size = f"HOT({hot_size})"
        elif ice_size:
            temp = 'ICE'
            size = f"ICE({ice_size})"

    return temp, size

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [76]:
all_items = []
driver = webdriver.Chrome(service=service)

for cat_name, menu_type in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    url = BASE_URL.format(menu_type)
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    menu_items = soup.find_all('li', class_=lambda x: x and 'animation' in x)
    print(f"  메뉴 수: {len(menu_items)}개")

    for i, li in enumerate(menu_items):
        try:
            name_tag = li.find('strong')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            eng_tag = li.find('p', class_='eng')
            eng_name = eng_tag.get_text(strip=True) if eng_tag else ''

            # 메뉴 클릭해서 팝업 열기
            a_tag = driver.find_elements(By.CSS_SELECTOR, 'li.animation a')[i]
            driver.execute_script("arguments[0].click();", a_tag)
            time.sleep(1)

            # 사이즈 수집
            temp, size = parse_size_from_popup(driver)

            # 닫기 버튼 클릭
            try:
                close_btn = driver.find_element(By.CSS_SELECTOR, 'button.close_b')
                driver.execute_script("arguments[0].click();", close_btn)
                time.sleep(1)
            except Exception as e:
                print(f"  닫기 버튼 에러: {e}")

            all_items.append({
                'brand': '매머드커피',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
            print(f"  [{i+1}] {name} | HOT:{temp} SIZE:{size}")

        except Exception as e:
            print(f"  [{i+1}] 에러: {e}")
            continue

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 32oz 수집 시작
  메뉴 수: 9개
  [1] 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [2] 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [3] 아샷추 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [4] 스노우 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [5] 허니베리 홍초 에이드 | HOT:ICE SIZE:ICE(32oz)
  [6] 패션 오렌지 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [7] 디카페인 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [8] 디카페인 아샷추 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [9] 디카페인 스노우 매머드 커피 | HOT:ICE SIZE:ICE(32oz)

▶ 커피 수집 시작
  메뉴 수: 37개
  [1] 아메리카노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [2] 꿀 커피 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [3] 아몬드 아메리카노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [4] 아샷추 아이스티 | HOT:ICE SIZE:ICE(20oz)
  [5] 카페 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [6] 카푸치노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [7] 꿀 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [8] 아몬드 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [9] 바닐라 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [10] 꿀바나 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [11] 카라멜 마키아토 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [12] 카페 모카 | 

In [78]:
df = pd.DataFrame(all_items)
df.to_csv('mammoth_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → mammoth_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → mammoth_menu.csv

카테고리별 수집 수:
category
커피          37
티·에이드       32
프라페·블렌디드    15
콜드브루        14
32oz         9
논커피          8
Name: count, dtype: int64

샘플 데이터:
   brand             name                            name_en     temp  \
0  매머드커피           매머드 커피             Mammoth Iced Americano      ICE   
1  매머드커피             아이스티                     Peach Iced Tea      ICE   
2  매머드커피         아샷추 아이스티           Peach Iced Tea(add shot)      ICE   
3  매머드커피       스노우 매머드 커피        Snow Mammoth Iced Americano      ICE   
4  매머드커피      허니베리 홍초 에이드        Honey Berry Red Vinegar Ade      ICE   
5  매머드커피      패션 오렌지 아이스티            Passion Orange Iced Tea      ICE   
6  매머드커피      디카페인 매머드 커피       Decaf Mammoth Iced Americano      ICE   
7  매머드커피    디카페인 아샷추 아이스티     Decaf Peach Iced Tea(add shot)      ICE   
8  매머드커피  디카페인 스노우 매머드 커피  Decaf Snow Mammoth Iced Americano      ICE   
9  매머드커피            아메리카노                          Americano  HOT/ICE   

                  size